# 作业 01：观察 MNIST 图片、像素矩阵与标签

本次先不训练模型。我们通过 **数字 0、3、7 三个真实样本**，观察图像输入 `x` 和标签 `y` 到底是什么。

完成后，你应能区分：

1. 一张手写数字图片和它的 **28 行 × 28 列像素矩阵**。
2. 原始的**整数类别标签**，例如 `7`。
3. 同一标签的 **one-hot 一维向量**，例如 `[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]`。

请从上到下运行全部单元格。首次运行需要联网下载 MNIST；后续会使用本地缓存。本作业只需要 CPU。


## 1. 准备环境与下载数据

沿用课程 Python 环境，需要 `torch`、`torchvision` 和 Jupyter。图片显示使用 `torchvision` 自带依赖 Pillow，无需额外安装绘图库。先确认当前 Notebook 使用的是你安装这些包的同一个环境。无需为本题重新安装或更换已经可用的 PyTorch。

下面使用 `torchvision.datasets.MNIST` 下载数据。文件放在当前工作目录下的 `data/`，此目录已被作业的 `.gitignore` 排除，**不需要上传数据集到 GitHub**。

这里不使用 `ToTensor()` 或归一化，先观察数据集中原始的 **0～255 整数灰度值**：0 是黑色，255 是白色，中间值是不同深浅的灰色。


In [1]:
from pathlib import Path
import sys
import torch
import torch.nn.functional as F
from torchvision import datasets
from PIL import Image
from IPython.display import display

print("当前 Python：", sys.executable)
print("PyTorch 版本：", torch.__version__)

DATA_DIR = Path("data")
train_dataset = datasets.MNIST(
    root=str(DATA_DIR),
    train=True,
    download=True,
    transform=None,  # 保留原始图片，不转换或归一化
)

print("数据目录：", DATA_DIR.resolve())
print("训练集样本数：", len(train_dataset))
print("全部图片 shape：", tuple(train_dataset.data.shape))
print("全部标签 shape：", tuple(train_dataset.targets.shape))
print("原始像素 dtype：", train_dataset.data.dtype)
print("原始标签 dtype：", train_dataset.targets.dtype)

ModuleNotFoundError: No module named 'torch'

## 2. 固定选择三个不同类别的真实样本

分别选择训练集中**第一个标签为 0、3、7 的样本**。这不是随机生成的矩阵，而是真实 MNIST 图像。

- 全部图片的 shape 是 `(60000, 28, 28)`：60000 个样本，每张 28×28。
- 取出一张图片后，像素矩阵 shape 是 `(28, 28)`。
- 本题的行列编号均从 **0** 开始：行 `00～27`、列 `00～27`。`x[r, c]` 表示第 r 行、第 c 列的像素。
- 此处还没有添加通道维度；之后送入 CNN 时，单张灰度图片通常表示为 `(1, 28, 28)`。


In [ ]:
import torch
from torchvision import datasets, transforms

transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)

selected_digits = [0, 3, 7]
sample_indices = {}
for digit in selected_digits:
    matches = torch.where(train_dataset.targets == digit)[0]
    sample_indices[digit] = int(matches[0].item())

for digit, index in sample_indices.items():
    print(f"数字 {digit}: 训练集索引 {index}")


NameError: name 'torch' is not defined

## 3. 完整显示图片、784 个像素值及两种标签表示

下面的函数会依次显示：图片、**全部 28 行 × 28 列整数值**、原始整数标签和 one-hot 向量。

矩阵使用逐行打印，不依赖张量的默认显示规则，因此不会把中间的数值省略成 `...`。第一行是列编号，左侧是行编号；如果屏幕较窄，请横向滚动输出区域查看右侧列。

one-hot 的**固定类别顺序是 `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]`**。长度 10 表示 10 个类别，不是 10 个样本。只有真实类别对应的位置为 1。


In [ ]:
def show_sample(digit):
    index = sample_indices[digit]
    x = train_dataset.data[index]             # uint8，shape=(28, 28)
    raw_label = train_dataset.targets[index]  # int64，标量张量，shape=()
    y_integer = int(raw_label.item())          # Python 整数
    y_one_hot = F.one_hot(raw_label, num_classes=10)

    assert tuple(x.shape) == (28, 28)
    assert x.dtype == torch.uint8
    assert y_integer == digit
    assert tuple(y_one_hot.shape) == (10,)
    assert int(y_one_hot.sum().item()) == 1
    assert int(y_one_hot.argmax().item()) == y_integer

    image, dataset_label = train_dataset[index]  # 原始 PIL 灰度图片及整数标签
    assert dataset_label == y_integer
    print(f"数字 {y_integer} 的图片（训练集索引 {index}）")
    print("仅按最近邻放大到 280×280 便于观察；原始数据仍是 28×28。")
    display(image.resize((280, 280), resample=Image.Resampling.NEAREST))

    print(f"【数字 {digit}：完整原始像素矩阵】")
    print(f"训练集索引 = {index}；shape = {tuple(x.shape)}；dtype = {x.dtype}")
    print("以下每行有 28 个像素值，共 28 行；范围 0～255。")
    print("列号 |" + "".join(f"{c:4d}" for c in range(28)))
    print("-----+" + "----" * 28)
    for r in range(28):
        values = "".join(f"{int(x[r, c].item()):4d}" for c in range(28))
        print(f"{r:02d}   |" + values)

    print(f"\n【数字 {digit}：同一个标签的两种表示】")
    print("数据集中的原始标签张量：", raw_label)
    print("原始标签张量 shape：", tuple(raw_label.shape))
    print("原始标签张量 dtype：", raw_label.dtype)
    print("整数标签（常数数值）：", y_integer)
    print("类别位置顺序：", list(range(10)))
    print("one-hot 一维向量：", y_one_hot.tolist())
    print("one-hot shape：", tuple(y_one_hot.shape))
    print(f"说明：类别 {y_integer} 对应的位置为 1，其余九个位置均为 0。")
    print("注意：这两个 y 表示同一个真实标签，不是模型预测的概率。")

### 例子 1：手写数字 0

这张图片的真实类别是 **数字 0**。MNIST 原始整数标签为 **`0`**，不是 one-hot 数组。

转换为 one-hot 后为 **`[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]`**，shape 为 `(10,)`。按从 0 开始的索引，**第 0 个位置为 1**。

运行下面的单元格，查看本例图片、完整矩阵以及实际打印的标签。

In [ ]:
show_sample(0)

### 例子 2：手写数字 3

这张图片的真实类别是 **数字 3**。MNIST 原始整数标签为 **`3`**，不是 one-hot 数组。

转换为 one-hot 后为 **`[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`**，shape 为 `(10,)`。按从 0 开始的索引，**第 3 个位置为 1**。

运行下面的单元格，查看本例图片、完整矩阵以及实际打印的标签。

In [ ]:
show_sample(3)

### 例子 3：手写数字 7

这张图片的真实类别是 **数字 7**。MNIST 原始整数标签为 **`7`**，不是 one-hot 数组。

转换为 one-hot 后为 **`[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]`**，shape 为 `(10,)`。按从 0 开始的索引，**第 7 个位置为 1**。

运行下面的单元格，查看本例图片、完整矩阵以及实际打印的标签。

In [ ]:
show_sample(7)

## 4. 对照总结

| 样本 | MNIST 原始整数标签 | one-hot 一维向量 | one-hot shape |
|---|---:|---|---|
| 数字 0 | `0` | `[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]` | `(10,)` |
| 数字 3 | `3` | `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]` | `(10,)` |
| 数字 7 | `7` | `[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]` | `(10,)` |

MNIST 实际保存的是整数类别标签；one-hot 是我们从这个编号转换得到的另一种表示。调用 PyTorch 的 `CrossEntropyLoss` 时，通常直接使用整数类别标签，无需先转成 one-hot。

### 请填写你的观察 (50分)

1. 从任意一个例子的完整矩阵中找出一个非零像素，记录它的行号、列号和数值，并指出图片中大致对应的位置。（15分）
2. 为什么一张图片的矩阵有 784 个像素值，而它的整数标签只需要一个数？（15分）
3. 数字 7 的 one-hot 为什么不是 `[7]`？向量长度 10 的含义是什么？（20分）

**我的回答：**

1. 选取 MNIST 数字 3 的样本：行号：10，列号：12，像素值：0.9922（原始灰度值 253，该像素位于整张 28×28 图像的中上区域，属于数字 3 的笔画轮廓区域
2. MNIST 图片是 28×28=784个像素，784 个像素是输入特征，而整数标签是这张图的分类目标（ground truth），只代表这张图片属于哪一个类别（0~9 中的哪一类），不需要描述细节
3. 如果使用 [7]，模型会错误认为类别之间存在天然的大小顺序；向量长度为 10，代表 MNIST 一共有 10 个独立类别，对应数字 0到9 向量的每一个位置分别对应一个类别。

### 保存与提交

填写观察后，重启内核并从上到下运行全部单元格，保存 Notebook 和输出。提交本文件；不要提交 `data/`、虚拟环境或 `.ipynb_checkpoints/`。
